In [28]:
# Model takes a list of sentences and outputs an array of score with a formality score fore each sentence.




In [29]:
# Imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# The word2vec imports

import gensim
import gensim.downloader as api
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# The LSTM imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import TransformerEncoder, TransformerEncoderLayer

In [57]:
class BidirectionalTransformer(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers, dropout=0.1):
        super(BidirectionalTransformer, self).__init__()
        
        # Positional encoding (optional if you're working with sequences)
        self.positional_encoding = nn.Embedding(5000, model_dim)  # You can customize the maximum length
        
        # Transformer encoder layers
        encoder_layer = TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dropout=dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layer, num_layers=num_layers)
    
        
    def forward(self, x):
        # Optionally add positional encoding (for sequences)
        seq_len, batch_size = x.size(0), x.size(1)
        positions = torch.arange(0, seq_len).unsqueeze(1).expand(seq_len, batch_size).to(x.device)
        x = x + self.positional_encoding(positions)
        
        # Transformer Encoder (bidirectional)
        encoded_output = self.transformer_encoder(x)
        
       
        
        return encoded_output
    
class FullyConnectedLayer(nn.Module):
    def __init__(self, input_size, output_size=1):
        super(FullyConnectedLayer, self).__init__()
        self.fc = nn.Linear(input_size, output_size)
    
    def forward(self, x):
        # Ensure the input is 1D
        if x.ndim != 1:
            raise ValueError("Input tensor must be 1D")
        
        # Check if the input size matches the expected input size (625 in this case)
        if x.size(0) != 625:  # Assuming input_size is 625
            raise ValueError("Input tensor must have length 625")

        output = self.fc(x)  # Apply the fully connected layer
        output = torch.sigmoid(output)  # Apply sigmoid activation
        return output


    

def sentences_to_vectors(sentences, model):
    encoding_dim = model.vector_size
    no_sentence = len(sentences)
    max_word_count = 50
    
    # Initialize a 3D array with zeros
    returned_array = np.zeros((encoding_dim, max_word_count, no_sentence))

    def tokenize(sentence):
        tokens = word_tokenize(sentence)  # Tokenize sentence
        return [word for word in tokens if word not in stopwords.words('english')]  # Remove stopwords

    for i, sentence in enumerate(sentences):
        words = tokenize(sentence)
        word_vectors = [model[word] for word in words if word in model]
        
        # Pad or truncate word_vectors to fit max_word_count
        if len(word_vectors) < max_word_count:
            # Pad with zeros if there are fewer than max_word_count word vectors
            padded_vectors = np.array(word_vectors + [[0] * encoding_dim] * (max_word_count - len(word_vectors)))
        else:
            # Truncate if there are more than max_word_count word vectors
            padded_vectors = np.array(word_vectors[:max_word_count])
        
        # Fill the 3D array
        returned_array[:, :, i] = padded_vectors.T  # Transpose to match shape (encoding_dim, max_word_count)

    return returned_array


def convert_column_sentences_to_vectors(df, column_name, model):
    encoding_dim = model.vector_size
    no_sentence = len(df[column_name])
    max_word_count = 100
    
    # Initialize a 3D array with zeros
    returned_array = np.zeros((encoding_dim, max_word_count, no_sentence))

    meaningless_words = {}  

    def tokenize(sentence):
        tokens = word_tokenize(sentence)  # Tokenize sentence
        return [word for word in tokens if word.lower() not in stopwords.words('english') and word.lower() not in meaningless_words]  # Remove stopwords and meaningless words

    for i, sentence in enumerate(df[column_name]):
        words = tokenize(sentence)
        word_vectors = [model[word] for word in words if word in model]
        
        # Pad or truncate word_vectors to fit max_word_count
        if len(word_vectors) < max_word_count:
            # Pad with zeros if there are fewer than max_word_count word vectors
            padded_vectors = np.array(word_vectors + [[0] * encoding_dim] * (max_word_count - len(word_vectors)))
        else:
            # Truncate if there are more than max_word_count word vectors
            padded_vectors = np.array(word_vectors[:max_word_count])
        
        # Fill the 3D array
        returned_array[:, :, i] = padded_vectors.T  # Transpose to match shape (encoding_dim, max_word_count)

    return returned_array

In [31]:
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [32]:
model = api.load('glove-twitter-25')

In [7]:
example_preprocessed_sentences = [
    "team wanted provide update latest progress marketing campaign",
    "successfully completed initial phase client happy results far",
    "however changes need addressed move forward",
    "arrange meeting next week go revised strategy ensure page",
    "please let know availability",
    "looking forward continued collaboration"
]

example_targets = np.array([0,1,1,1,0,0])



In [41]:
gpt_sentences = [
    "like express gratitude assistance",
    "thanks help",
    "regret inform application unsuccessful",
    "hey wanted say thanks",
    "please hesitate reach questions",
    "worries feel free ask",
    "meeting take place 10 am friday",
    "meet 10 friday",
    "pleased announce launch new product",
    "excited share new product",
    "hope email finds well",
    "hope doing great",
    "find attached report requested",
    "check report get chance",
    "appreciate understanding matter",
    "thanks understanding",
    "presence meeting greatly appreciated",
    "wait see meeting",
    "proposal requires immediate attention",
    "heads up proposal",
    "looking forward response",
    "let know think",
    "results analysis available",
    "analysis ready check",
    "like schedule followup meeting",
    "followup meeting next week",
    "thank prompt reply",
    "thanks getting back quickly",
    "please ensure documents submitted time",
    "make sure send docs time",
    "kindly request cooperation matter",
    "appreciate help",
    "message intended designated recipient",
    "note eyes",
    "feedback invaluable",
    "love hear thoughts",
    "committed providing excellent service",
    "great service",
    "inquiries hesitate ask",
    "questions let know",
    "financial report released next week",
    "financials drop next week",
    "please review attached document carefully",
    "take look attached doc",
    "appreciate attention matter",
    "thanks looking",
    "inform request processed",
    "letting request done",
    "here assist issues",
    "here help need",
    "satisfaction priority",
    "want happy service",
    "thank continued support",
    "thanks sticking",
    "notify once process complete",
    "let know set",
    "reminder upcoming appointment",
    "forget appointment coming",
    "value input suggestions",
    "love hear ideas",
    "cooperation greatly appreciated",
    "thanks help",
    "correspondence serves clarify situation",
    "quick note clear things",
    "conducting maintenance system tonight",
    "heads doing maintenance tonight",
    "thank understanding time",
    "appreciate patience",
    "let know need assistance",
    "need else holler",
    "excited announce new initiative",
    "thrilled share new initiative",
    "confirm appointment",
    "confirming appointment",
    "contribution greatly valued",
    "appreciate input",
    "thank bringing attention",
    "thanks flagging",
    "look forward working",
    "wait work together",
    "email confidential intended recipient",
    "between confidential",
    "strive exceed expectations",
    "aim impress",
    "kindly request review guidelines",
    "check guidelines get chance",
    "trust appreciated",
    "thanks trusting",
    "inform changes made",
    "wanted update changes",
    "committed maintaining high standards",
    "great keeping top-notch",
    "thank cooperation",
    "thanks help",
    "would like hear",
    "let know thoughts",
    "prompt attention matter appreciated",
    "love quick feedback"
]



In [242]:
gpt_ground_truth = np.array([
    1,  # "I would like to express my gratitude for your assistance."
    0,  # "Thanks for your help!"
    1,  # "We regret to inform you that your application has been unsuccessful."
    0,  # "Hey, just wanted to say thanks!"
    1,  # "Please do not hesitate to reach out if you have any questions."
    0,  # "No worries, feel free to ask me anything!"
    1,  # "The meeting will take place at 10 AM on Friday."
    0,  # "Let's meet at 10 on Friday."
    1,  # "We are pleased to announce the launch of our new product."
    0,  # "Super excited to share our new product!"
    1,  # "I hope this email finds you well."
    0,  # "What's up? Hope you're doing great!"
    1,  # "Please find attached the report you requested."
    0,  # "Can you check out this report when you get a chance?"
    1,  # "We appreciate your understanding in this matter."
    0,  # "Thanks for being so understanding!"
    1,  # "Your presence at the meeting is greatly appreciated."
    0,  # "Can't wait to see you at the meeting!"
    1,  # "This proposal requires your immediate attention."
    0,  # "Just a heads up about the proposal!"
    1,  # "We are looking forward to your response."
    0,  # "Let me know what you think!"
    1,  # "The results of the analysis are available now."
    0,  # "The analysis is ready. Check it out!"
    1,  # "We would like to schedule a follow-up meeting."
    0,  # "How about a follow-up meeting next week?"
    1,  # "Thank you for your prompt reply."
    0,  # "Thanks for getting back to me so quickly!"
    1,  # "Please ensure that all documents are submitted on time."
    0,  # "Make sure to send all the docs on time, okay?"
    1,  # "We kindly request your cooperation in this matter."
    0,  # "We appreciate your help with this!"
    1,  # "This message is intended for the designated recipient only."
    0,  # "Just a note for you, it's for your eyes only!"
    1,  # "Your feedback is invaluable to us."
    0,  # "We'd love to hear your thoughts!"
    1,  # "We are committed to providing excellent service."
    0,  # "We're all about great service here!"
    1,  # "Should you have any inquiries, please do not hesitate to ask."
    0,  # "If you have any questions, just let me know!"
    1,  # "The financial report will be released next week."
    0,  # "The financials will drop next week!"
    1,  # "Please review the attached document carefully."
    0,  # "Take a look at the attached doc when you can."
    1,  # "We appreciate your attention to this matter."
    0,  # "Thanks for looking into this!"
    1,  # "This is to inform you that your request has been processed."
    0,  # "Just letting you know, your request is done!"
    1,  # "We are here to assist you with any issues."
    0,  # "We're here to help if you need anything."
    1,  # "Your satisfaction is our priority."
    0,  # "We want you to be happy with our service!"
    1,  # "Thank you for your continued support."
    0,  # "Thanks for sticking with us!"
    1,  # "We will notify you once the process is complete."
    0,  # "We'll let you know when it's all set!"
    1,  # "This is a reminder for your upcoming appointment."
    0,  # "Don't forget about your appointment coming up!"
    1,  # "We value your input and suggestions."
    0,  # "We’d love to hear your ideas!"
    1,  # "Your cooperation is greatly appreciated."
    0,  # "Thanks a lot for your help!"
    1,  # "This correspondence serves to clarify the situation."
    0,  # "Just a quick note to clear things up."
    1,  # "We will be conducting maintenance on the system tonight."
    0,  # "Heads up, we're doing maintenance tonight."
    1,  # "Thank you for your understanding during this time."
    0,  # "Appreciate your patience with this!"
    1,  # "Please let us know if you need any further assistance."
    0,  # "If you need anything else, just holler!"
    1,  # "We are excited to announce a new initiative."
    0,  # "Thrilled to share a new initiative with you!"
    1,  # "This is to confirm your appointment."
    0,  # "Just confirming our appointment!"
    1,  # "Your contribution is greatly valued."
    0,  # "We really appreciate your input!"
    1,  # "Thank you for bringing this to our attention."
    0,  # "Thanks for flagging that!"
    1,  # "We look forward to working with you."
    0,  # "Can't wait to work together!"
    1,  # "This email is confidential and intended for the recipient only."
    0,  # "Just between us, this is confidential."
    1,  # "We strive to exceed your expectations."
    0,  # "We aim to impress you!"
    1,  # "We kindly request that you review the guidelines."
    0,  # "Please check the guidelines when you get a chance."
    1,  # "Your trust in us is appreciated."
    0,  # "Thanks for trusting us!"
    1,  # "This is to inform you of the changes made."
    0,  # "Just wanted to update you on the changes."
    1,  # "We are committed to maintaining high standards."
    0,  # "We're all about keeping it top-notch!"
    1,  # "Thank you for your cooperation."
    0,  # "Thanks for your help!"
    1,  # "We would like to hear from you."
    0,  # "Let us know your thoughts!"
    1,  # "Your prompt attention to this matter is appreciated."
    0   # "Would love your quick feedback on this!"
])
targets_tensor = torch.from_numpy(gpt_ground_truth).float()


X_train, X_test, y_train, y_test = train_test_split(
    gpt_sentences,  # Features: indices of the sentences
    gpt_ground_truth,                  # Labels: ground truth
    test_size=0.3,                     # 20% for testing
    random_state=422                   # For reproducibility
)

In [243]:
sentence_vectors_train = sentences_to_vectors(X_train, model)
sentence_vectors_test = sentences_to_vectors(X_test, model)


In [244]:
# LSTM hyperparameters 

input_dim = 25       # Example input dimension (e.g., number of features)
model_dim = 25       # Dimension of the model (embedding size)
num_heads = 1        # Number of attention heads
num_layers = 1       # Number of transformer layers
dropout = 0.3       # Dropout rate

fc_dim = 625


# Training hyperparameters 

sequence_length = 1  # Each sample is a single time step
batch_size = sentence_vectors_train.shape[1]  # All samples in one batch
input_size = sentence_vectors_train.shape[0]  # Number of features

In [245]:
lstm = BidirectionalTransformer(input_dim, model_dim, num_heads, num_layers, dropout)
fc_layer =  FullyConnectedLayer(fc_dim)

c:\Users\timur\Documents\GitHub\EmailSentin\env\Lib\site-packages\torch\nn\modules\transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [246]:
# Training Hyperparameters

num_epochs = 450   # Number of training epochs

# Optimizer Hyperparameters

opt_lr = 0.0001



In [247]:
# The training process!
lstm.train()
criterion = nn.BCELoss()
optimizer = optim.Adam(lstm.parameters(), lr=opt_lr)



for epoch in range(num_epochs):
    output_preds_train = np.array([])
    optimizer.zero_grad()  # Clear previous gradients
    
    # Initialize total_epoch_loss as a float
    total_epoch_loss = torch.tensor(0.0, requires_grad=True)

    for sen_ix in range(sentence_vectors_train.shape[2]):
        sentence_array = sentence_vectors_train[:, :, sen_ix]
        input_tensor = torch.from_numpy(sentence_array).float().view(sequence_length, batch_size, input_size)
        output_tensor = lstm(input_tensor)  # Get output from the transformer
        
        # Use the last time step's output directly
        output = fc_layer(torch.flatten(output_tensor))  # Use the last time step’s output
        
        # Flatten output if needed
        output = output.view(-1)  
        

        # Ensure targets_tensor is properly shaped
        target_tensor = torch.tensor(y_train[sen_ix], dtype=torch.float32).unsqueeze(0)  # Convert target to tensor

        
        
        
        # Compute loss
        sentence_loss = criterion(output, target_tensor)  
        output = output.detach().numpy()
        output_preds_train = np.append(output_preds_train,output)

        output_preds_01_train = np.array([1 if output > 0.5 else 0 for output in output_preds_train])
        
        # Accumulate loss
        total_epoch_loss = total_epoch_loss + sentence_loss

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {total_epoch_loss:.4f}")
    total_epoch_loss.backward()  # Backpropagate
    optimizer.step()  # Update parameters


Epoch [1/450], Loss: 46.0737
Epoch [2/450], Loss: 43.7215
Epoch [3/450], Loss: 47.4876
Epoch [4/450], Loss: 45.0919
Epoch [5/450], Loss: 47.0029
Epoch [6/450], Loss: 45.8526
Epoch [7/450], Loss: 44.9928
Epoch [8/450], Loss: 46.9842
Epoch [9/450], Loss: 44.2239
Epoch [10/450], Loss: 45.4013
Epoch [11/450], Loss: 44.5062
Epoch [12/450], Loss: 45.2844
Epoch [13/450], Loss: 45.5127
Epoch [14/450], Loss: 43.9147
Epoch [15/450], Loss: 44.5273
Epoch [16/450], Loss: 42.5423
Epoch [17/450], Loss: 45.8133
Epoch [18/450], Loss: 45.5820
Epoch [19/450], Loss: 43.3339
Epoch [20/450], Loss: 43.1996
Epoch [21/450], Loss: 43.6682
Epoch [22/450], Loss: 44.3008
Epoch [23/450], Loss: 45.3739
Epoch [24/450], Loss: 42.8665
Epoch [25/450], Loss: 42.2415
Epoch [26/450], Loss: 42.9314
Epoch [27/450], Loss: 43.4931
Epoch [28/450], Loss: 43.2636
Epoch [29/450], Loss: 43.8160
Epoch [30/450], Loss: 43.8532
Epoch [31/450], Loss: 41.3732
Epoch [32/450], Loss: 42.5972
Epoch [33/450], Loss: 41.9963
Epoch [34/450], Los

In [248]:
lstm.eval()

output_preds = np.array([])


# Forward pass to get the output
for sen_ix in range(sentence_vectors_test.shape[2]):
        sentence_array = sentence_vectors_test[:, :, sen_ix]
        input_tensor = torch.from_numpy(sentence_array).float().view(sequence_length, batch_size, input_size)
        output_tensor = lstm(input_tensor)  # Get output from the transformer
        
        # Use the last time step's output directly
        output = fc_layer(torch.flatten(output_tensor))  # Use the last time step’s output
        
        # Flatten output if needed
        output = output.detach().numpy()
        output_preds = np.append(output_preds,output)

        output_preds_01 = np.array([1 if output > 0.5 else 0 for output in output_preds])


        

In [249]:
output_preds_01

array([1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 1])

In [250]:
y_test

array([1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 1])

In [251]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test , output_preds_01)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.73


In [252]:
output_preds_01_train

array([0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0,
       0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0,
       1, 1])

In [253]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_train , output_preds_01_train)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.91


In [58]:
# Here we load the ready data.

data_dir = r"C:\Users\timur\Documents\GitHub\EmailSentin\data\ready_data.csv"
df_processed_data = pd.read_csv(data_dir)

mapping = {
    'Very Informal': -2,
    'Informal': -1,
    'Neutral': 0,
    'Formal': 1,
    'Very Formal': 2
}


# Encoding the sentences and extracting ground truth array. 

encoded_matrix = convert_column_sentences_to_vectors(df_processed_data, 'text', model)
# Convert the label column to a NumPy array
df_processed_data['numerical_label'] = df_processed_data['label'].map(mapping)
ground_truth = df_processed_data['numerical_label']

# The train test split.

encoded_matrix_transposed = np.transpose(encoded_matrix, (2, 0, 1))  # Rearrange to shape (1333, 25, 25)

# Perform a train-test split
X_train, X_test, y_train, y_test = train_test_split(
    encoded_matrix_transposed, 
    ground_truth, 
    test_size=0.2,  # 20% for testing
    random_state=42,  # For reproducibility
    shuffle=True  # Shuffle the samples before splitting
)

In [27]:
encoded_matrix

(1066, 25, 25)

In [15]:
ground_truth 

array(['Formal', 'Neutral', 'Informal', ..., 'Very Informal', 'Informal',
       'Very Informal'], dtype=object)

In [49]:
encoded_matrix.shape

(25, 100, 1333)

In [69]:
encoded_matrix[:,5,2]

array([-1.63030005,  0.38299999,  1.0927    ,  0.64552999,  0.31498   ,
       -0.89894003,  1.08430004, -0.80362999,  0.06931   ,  0.86268997,
       -0.12375   ,  0.0060227 , -2.88260007, -0.38343   ,  0.1866    ,
        0.19399001,  1.3003    , -0.22066   ,  0.12759   , -0.089674  ,
        1.40849996,  0.19228999, -0.028555  , -1.12380004, -0.29025999])

In [66]:
df_processed_data.loc[2]['text']

"Congratulations on your new adventure.  I'll miss having you as a neighbor and a colleague.Kay"